RunnableLambda

사용자 정의 함수를 정의하고 실행 실행할 수 있음

In [9]:
from dotenv import load_dotenv
from operator import itemgetter
import json

from langchain_teddynote import logging
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnableConfig
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain.callbacks import get_openai_callback

In [2]:
load_dotenv()

True

In [ ]:
logging.langsmith("langchain-LCEL-Advanced")

In [3]:
# 텍스트의 길이를 반환하는 함수
def length_function(text):
    return len(text)

# 두 텍스트의 길이를 곱하는 함수
def _multiple_length_function(text1, text2):
    return len(text1) * len(text2)

# 딕셔너리에서 "text1"과 "text2"의 길이를 곱하는 함수
def multiple_length_function(_dict):  # 2개 인자를 받는 함수로 연결하는 wrapper 함수
    return _multiple_length_function(_dict["text1"], _dict["text2"])

In [4]:
prompt = ChatPromptTemplate.from_template("what is {a} + {b}?")

In [5]:
model = ChatOpenAI()

In [ ]:
chain = (
    {
        "a": itemgetter("input_1") | RunnableLambda(length_function), 
        "b": {"text1": itemgetter("input_1"), "text2": itemgetter("input_2")} | RunnableLambda(multiple_length_function),
    } 
    | prompt 
    | model 
    | StrOutputParser()
)

In [7]:
chain.invoke({"input_1": "bar", "input_2": "gah"})

'3 + 9 is equal to 12.'

RunnerConfig 활용
- chain의 실행을 제어하거나 실행 환경에 필요한 추가 정보를(콜백, 태그 및 기타 구성 정보) 중첩된 실행에 전달할 수 있음

In [10]:
# 입력된 텍스트에 오류가 있을 경우 오류 메시지를 전달하고 내용을 자동으로 수정하는 함수
def parse_or_fix(text: str, config: RunnableConfig):
    # 다음 텍스트를 수정하기 위한 프롬프트
    fixing_prompt = ChatPromptTemplate.from_template(
        "Fix the following text:\n\ntext\n{input}\n\nError: {error}"
        " Don't narrate, just respond with the fixed data."
    )
    
    # 수정 체인
    fixing_chain = (fixing_prompt | ChatOpenAI() | StrOutputParser())
    
    for _ in range(3):  # 최대 3번 시도
        try:
            return json.loads(text)  # JSON 형식으로 텍스트를 파싱
        except Exception as e:
            # 파싱 중 오류가 발생하면 수정 체인을 호출하여 텍스트를 수정
            text = fixing_chain.invoke({"input": text, "error": e}, config)
            print(f"config: {config}")
    
    # 3번 시도했는데도, 파싱에 실패하면 "Failed to parse" 문자열 반환
    return "Failed to parse"

In [11]:
with get_openai_callback() as cb:
    # RunnableLambda로 parse_or_fix 함수 호출
    output = RunnableLambda(parse_or_fix).invoke(
        input="{foo:: bar}", 
        config={"tags": ["my-tag"], "callbacks": [cb]},  # config 를 전달
    )
    
    # 수정한 결과 출력
    print(f"\n\n수정한결과:\n{output}")

config: {'tags': ['my-tag'], 'metadata': {}, 'callbacks': <langchain_core.callbacks.manager.CallbackManager object at 0x000001F0B2CBA410>, 'recursion_limit': 25, 'configurable': {}}


수정한결과:
{'foo': 'bar'}


In [12]:
output

{'foo': 'bar'}